# Data Wrangling using Pandas

## Dataset Description: Google Books
This data was acquired from Google Books store. Google API was used to acquire the data. Nine features were gathered for each book in the data set. the column names mostly are self explanatory nevertheless, it will be explained below.

- title : the title of the book.
- authors : name of the authors of the books (might include more than one author).
- language : the language of the book
- generes\categories : the categories associated with the book (by Google store)
- rating\averageRating : the average rating of each book out of 5.
- maturityRating : wheather the content of the book is for mature or NOT MATURE audience.
- publisher : the name of the publisher.
- publishedDate : when the book was published.
- pageCount : number of pages of the books.
- voters : the number of voters to the book.
- ISBN : the unique identifier for each book.
- description : brief introductory description of the book.
- price : price of the book on the google books store
- currency : the currency of the price in the google books store.

### Tasks:
- Load the dataset
  - Load google_books_1299.csv into a pandas DataFrame.
- Analyze genre distribution
  - Display the value counts of the generes column to understand the distribution of book genres.
- Process genres
  - Split the generes column by the comma (,) delimiter.
  - Compute the frequency of each genre.
  - Retain the top 10 most frequent genres, and replace all other genres with Other.
  - If multiple Other genres appear in the same row, retain only one Other.
- Explode genres
  - Transform the generes column so that each row contains exactly one genre.
  - Example: A row with generes of Fiction, Mystery will be split into two rows: one with Fiction and the other with Mystery. Refer to the df.explode() documentation for - guidance.
- Hyphenate ISBN numbers
  - Use the isbnid library to hyphenate the ISBN numbers in the ISBN column. Refer to the isbn.ipynb notebook as an example. Skip any records with invalid ISBN numbers.
- Extract ISBN components
  - Create two new columns:
    - registration_group: the second part of the hyphenated ISBN.
    - publisher_code: the third part of the hyphenated ISBN.
    - Example: For ISBN 978-1-61262-686-4, registration_group is 1 and publisher_code is 61262.
- Create a pivot table of ratings
  - Generate a pivot table where:
    - Rows correspond to registration_group.
    - Columns correspond to generes (including Other).
    - Values are the average rating for each combination, rounded to two decimal places.
    - If no ratings exist for a particular combination, fill the value with 0.

### Setup Code (Please run this first to set up the environment)

In [1]:
pip install isbnid


[notice] A new release of pip is available: 25.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


If you haven't install `isbnid` yet, you may use the following command:
```bash
pip install isbnid
```

In [1]:
import numpy as np
import pandas as pd
import isbn
from collections import Counter, defaultdict

In [2]:
if __name__ == "__main__":
    CSV_PATH = 'google_books_1299.csv'

### Load Dataset

In [3]:
def load_data(csv_path):
    """
    Load the Google Books dataset from a CSV file into a Pandas DataFrame.
    IN: csv_path, str, path to the CSV file
    OUT: google_books_df, pd.DataFrame
    """
    # Your Code Here
    google_books_df = pd.read_csv(csv_path)
    return google_books_df

In [4]:
if __name__ == "__main__":
    google_books_df = load_data(CSV_PATH)
    display(google_books_df)

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192.0,none,9781612626864,English,"Jul 31, 2014"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,"Fiction , Mystery &amp, Detective , Cozy , Gen...",9780758272799,English,"Jul 1, 2007"
2,2,The Art of Super Mario Odyssey,Nintendo,3.9,9,133.85,SAR,Take a globetrotting journey all over the worl...,Dark Horse Comics,368.0,"Games &amp, Activities , Video &amp, Electronic",9781506713816,English,"Nov 5, 2019"
3,3,Getting Away Is Deadly: An Ellie Avery Mystery,Sara Rosett,4.0,10,26.15,SAR,"With swollen feet and swelling belly, pregnant...",Kensington Publishing Corp.,320.0,none,9781617734076,English,"Mar 1, 2009"
4,4,"The Painted Man (The Demon Cycle, Book 1)",Peter V. Brett,4.5,577,28.54,SAR,The stunning debut fantasy novel from author P...,HarperCollins UK,544.0,"Fiction , Fantasy , Dark Fantasy",9780007287758,English,"Jan 8, 2009"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,746,The Complete Art of War,Sun Tzu,4.7,110,7.84,SAR,Collected here in this 4-in-1 omnibus are the ...,Simon and Schuster,667.0,"Political Science , Political Ideologies , Gen...",9781627931502,English,"Apr 29, 2013"
747,747,Ready Player One,Ernest Cline,4.6,"17,719",39.34,SAR,Nominated as one of America’s best-loved novel...,Broadway Books,384.0,"Fiction , Dystopian",9780307887450,English,"Aug 16, 2011"
748,748,Edgedancer: From the Stormlight Archive,Brandon Sanderson,4.8,221,36.79,SAR,From #1 New York Times bestselling author Bran...,Tor Books,226.0,none,9781250166609,English,"Oct 17, 2017"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,"Comics & Graphic Novels , Science Fiction",Original pages,English,"May 22, 2019"


### Analyze genre distribution

In [5]:
def get_genre_distribution(df):
    """
    Display the value counts of the generes column
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: genre_counts, dict, dictionary with genres as keys and their frequencies as values
    """
    # Your Code Here
    genre_counts = df['generes'].value_counts().to_dict() # genre column is misspelled in the csv file as generes
    
    return genre_counts

In [6]:
if __name__ == "__main__":
    genre_distribution = get_genre_distribution(google_books_df.copy())
    for genre, count in genre_distribution.items():
        print(f"{genre}: {count}")

none: 315
Fiction , Thrillers , Suspense: 21
Comics &amp, Graphic Novels , Superheroes: 19
Fiction , Fantasy , Epic: 18
Fiction , Mystery &amp, Detective , General: 11
Fiction , Action &amp, Adventure: 10
Comics & Graphic Novels , Science Fiction: 10
Comics & Graphic Novels , General: 10
Fiction , Science Fiction , Action &amp, Adventure: 9
Fiction , Thrillers , Crime: 9
Fiction , Mystery &amp, Detective , Amateur Sleuth: 9
Fiction , Mystery &amp, Detective , Cozy , General: 9
Fiction , Fantasy , General: 9
Self-Help , Personal Growth , Success: 8
Fiction , Fantasy , Action &amp, Adventure: 8
Fiction , Fantasy , Dragons &amp, Mythical Creatures: 7
Fiction , Mystery &amp, Detective , Women Sleuths: 7
Fiction , Classics: 7
Business &amp, Economics , Motivational: 6
Business & Economics , Mentoring & Coaching: 6
Self-Help , Personal Growth , General: 6
Fiction , General: 6
Business &amp, Economics , General: 5
Fiction , Mystery &amp, Detective , Private Investigators: 5
Juvenile Fiction ,

### Process genres

- Since the missing values in genres column were filled with string `none`, ensure that `none` is replaced by na, and not counted as a genre.
- You may want to replace `&amp,` with `&`. Else it may cause mis-splitting of genres.
- Ensure that if multiple 'Other' genres appear in the same row, only one 'Other' is retained.

In [10]:
def process_genres(df):
    """
    Process the generes column
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: df, pd.DataFrame, dataframe with processed generes
    """

    # TODO: Assuming the grader is using generes instead of genres, keep as generes for now.

    # Replace 'none' with NaN
    # Your Code Here
    df['generes'] = df['generes'].replace('none', pd.NA)

    # Replace '&amp,' with '&'
    # Your Code Here
    df['generes'] = df['generes'].str.replace('&amp,', '&', regex=False)

    # Split the generes column by the comma (,) delimiter
    # Your Code Here
    df['generes'] = df['generes'].str.split(',')
    
    # Compute the frequency of each genre
    # Your Code Here
    cleaned_series = df['generes'].apply(lambda x: [g.strip() for g in x] if isinstance(x, list) else x)
    genre_counts = cleaned_series.explode().value_counts()
    
    # Retain the top 10 most frequent genres
    # Your Code Here
    top_genres = set(genre_counts.head(10).index)
    
    # Replace all other genres with Other
    # Your Code Here
    df['generes'] = df['generes'].apply(
        lambda x: list(dict.fromkeys([g.strip() if g.strip() in top_genres else 'Other' for g in x])) 
        if isinstance(x, list) else x
    )
    
    return df

In [11]:
if __name__ == "__main__":
    google_books_processed_genres = process_genres(google_books_df.copy())
    display(google_books_processed_genres)

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192.0,<NA>,9781612626864,English,"Jul 31, 2014"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,"[Fiction, Mystery & Detective, Other, General]",9780758272799,English,"Jul 1, 2007"
2,2,The Art of Super Mario Odyssey,Nintendo,3.9,9,133.85,SAR,Take a globetrotting journey all over the worl...,Dark Horse Comics,368.0,[Other],9781506713816,English,"Nov 5, 2019"
3,3,Getting Away Is Deadly: An Ellie Avery Mystery,Sara Rosett,4.0,10,26.15,SAR,"With swollen feet and swelling belly, pregnant...",Kensington Publishing Corp.,320.0,<NA>,9781617734076,English,"Mar 1, 2009"
4,4,"The Painted Man (The Demon Cycle, Book 1)",Peter V. Brett,4.5,577,28.54,SAR,The stunning debut fantasy novel from author P...,HarperCollins UK,544.0,"[Fiction, Fantasy, Other]",9780007287758,English,"Jan 8, 2009"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
746,746,The Complete Art of War,Sun Tzu,4.7,110,7.84,SAR,Collected here in this 4-in-1 omnibus are the ...,Simon and Schuster,667.0,"[Other, General]",9781627931502,English,"Apr 29, 2013"
747,747,Ready Player One,Ernest Cline,4.6,"17,719",39.34,SAR,Nominated as one of America’s best-loved novel...,Broadway Books,384.0,"[Fiction, Other]",9780307887450,English,"Aug 16, 2011"
748,748,Edgedancer: From the Stormlight Archive,Brandon Sanderson,4.8,221,36.79,SAR,From #1 New York Times bestselling author Bran...,Tor Books,226.0,<NA>,9781250166609,English,"Oct 17, 2017"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,"[Comics & Graphic Novels, Science Fiction]",Original pages,English,"May 22, 2019"


### Explode genres

In [13]:
def explode_genres(df):
    """
    Transform the generes column so that each row contains exactly one genre
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: df, pd.DataFrame, dataframe with exploded generes
    """
    df = df.explode('generes')
    return df

In [14]:
if __name__ == "__main__":
    google_books_exploded = explode_genres(google_books_processed_genres.copy())
    display(google_books_exploded)

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192.0,<NA>,9781612626864,English,"Jul 31, 2014"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Fiction,9780758272799,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Mystery & Detective,9780758272799,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Other,9780758272799,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,General,9780758272799,English,"Jul 1, 2007"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,747,Ready Player One,Ernest Cline,4.6,"17,719",39.34,SAR,Nominated as one of America’s best-loved novel...,Broadway Books,384.0,Other,9780307887450,English,"Aug 16, 2011"
748,748,Edgedancer: From the Stormlight Archive,Brandon Sanderson,4.8,221,36.79,SAR,From #1 New York Times bestselling author Bran...,Tor Books,226.0,<NA>,9781250166609,English,"Oct 17, 2017"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Comics & Graphic Novels,Original pages,English,"May 22, 2019"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Science Fiction,Original pages,English,"May 22, 2019"


### Hyphenate ISBN numbers

- Be careful that some ISBN values are not valid. For non-valid ISBN values, you can convert them to NaN.
- A typical usage of `isbnid` library is as follows:
    ```python
    import isbn # note that the import is 'isbn', not 'isbnid'
    isbn_val = isbn.ISBN('9781612626864')
    isbn_val.hyphen()
    ```

In [15]:
def hyphenate_isbn(df):
    """
    Hyphenate the ISBN numbers in the ISBN column
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: df, pd.DataFrame, dataframe with hyphenated ISBNs
    """
    # Your Code Here
    def to_hyphen(value):
        try:
            return isbn.ISBN(str(value)).hyphen() # e.g. 9781612626864 -> 978-1-61262-686-4
        except Exception:
            return np.nan # invalid ISBNs (e.g. 'Flowing text') become NaN

    df['ISBN'] = df['ISBN'].apply(to_hyphen)
    return df

In [16]:
if __name__ == "__main__":
    google_books_hyphenated = hyphenate_isbn(google_books_exploded.copy())
    display(google_books_hyphenated)

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192.0,<NA>,978-1-61262-686-4,English,"Jul 31, 2014"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Fiction,978-0-7582-7279-9,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Mystery & Detective,978-0-7582-7279-9,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Other,978-0-7582-7279-9,English,"Jul 1, 2007"
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,General,978-0-7582-7279-9,English,"Jul 1, 2007"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,747,Ready Player One,Ernest Cline,4.6,"17,719",39.34,SAR,Nominated as one of America’s best-loved novel...,Broadway Books,384.0,Other,978-0-307-88745-0,English,"Aug 16, 2011"
748,748,Edgedancer: From the Stormlight Archive,Brandon Sanderson,4.8,221,36.79,SAR,From #1 New York Times bestselling author Bran...,Tor Books,226.0,<NA>,978-1-250-16660-9,English,"Oct 17, 2017"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Comics & Graphic Novels,NaN,English,"May 22, 2019"
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Science Fiction,NaN,English,"May 22, 2019"


### Extract ISBN components

- Make sure to skip rows with invalid ISBN numbers.

In [18]:
def extract_isbn_components(df):
    """
    Create two new columns: registration_group and publisher_code
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: df, pd.DataFrame, dataframe with new ISBN component columns
    """
    # Mask to filter out rows with valid ISBNs
    # Your Code Here
    valid_mask = df['ISBN'].notna() # invalid ISBNs were set to NaN in hyphenate_isbn

    # Update registration_group and publisher_code only for valid ISBNs
    # Your Code Here
    isbn_parts = df.loc[valid_mask, 'ISBN'].str.split('-') # e.g. 978-1-61262-686-4 -> ['978', '1', '61262', '686', '4']
    df.loc[valid_mask, 'registration_group'] = isbn_parts.str[1]
    df.loc[valid_mask, 'publisher_code'] = isbn_parts.str[2]

    return df

In [19]:
if __name__ == "__main__":
    google_books_with_isbn_components = extract_isbn_components(google_books_hyphenated.copy())
    display(google_books_with_isbn_components)

,Unnamed: 0,title,author,rating,voters,price,currency,description,publisher,page_count,generes,ISBN,language,published_date,registration_group,publisher_code
0,0,Attack on Titan: Volume 13,Hajime Isayama,4.6,428,43.28,SAR,NO SAFE PLACE LEFT At great cost to the Garris...,Kodansha Comics,192.0,<NA>,978-1-61262-686-4,English,"Jul 31, 2014",1,61262
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Fiction,978-0-7582-7279-9,English,"Jul 1, 2007",0,7582
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Mystery & Detective,978-0-7582-7279-9,English,"Jul 1, 2007",0,7582
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,Other,978-0-7582-7279-9,English,"Jul 1, 2007",0,7582
1,1,Antiques Roadkill: A Trash 'n' Treasures Mystery,Barbara Allan,3.3,23,26.15,SAR,Determined to make a new start in her quaint h...,Kensington Publishing Corp.,288.0,General,978-0-7582-7279-9,English,"Jul 1, 2007",0,7582
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
747,747,Ready Player One,Ernest Cline,4.6,"17,719",39.34,SAR,Nominated as one of America’s best-loved novel...,Broadway Books,384.0,Other,978-0-307-88745-0,English,"Aug 16, 2011",0,307
748,748,Edgedancer: From the Stormlight Archive,Brandon Sanderson,4.8,221,36.79,SAR,From #1 New York Times bestselling author Bran...,Tor Books,226.0,<NA>,978-1-250-16660-9,English,"Oct 17, 2017",1,250
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Comics & Graphic Novels,NaN,English,"May 22, 2019",NaN,NaN
749,749,Team Sonic Racing Plus Deluxe Turbo Championsh...,Caleb Goellner,4.8,30,11.78,SAR,When the evil Dr. Eggman invites himself to Do...,IDW Publishing,52.0,Science Fiction,NaN,English,"May 22, 2019",NaN,NaN


### Create a pivot table of ratings

- Round the values to two decimal places.

In [ ]:
def create_rating_pivot_table(df):
    """
    Generate a pivot table of ratings
    IN: df, pd.DataFrame, dataframe containing book information
    OUT: pivot_table, pd.DataFrame, pivot table of ratings
    """
    # Your Code Here
    return pivot_table

In [ ]:
if __name__ == "__main__":
    rating_pivot_table = create_rating_pivot_table(google_books_with_isbn_components.copy())
    display(rating_pivot_table)

### DataFrame Schema

In [ ]:
if __name__ == "__main__":
    google_books_processed_genres.info()

In [ ]:
if __name__ == "__main__":
    google_books_exploded.info()

In [ ]:
if __name__ == "__main__":
    google_books_hyphenated.info()

In [ ]:
if __name__ == "__main__":
    google_books_with_isbn_components.info()

In [ ]:
if __name__ == "__main__":
    rating_pivot_table.info()